# 01 — Update & compound IDX TF15 parquet

Download rolling TF15 Yahoo Finance, keep only completed candles, then merge it
into the existing historical parquet by `(ticker, date)`. The write is atomic,
so an interrupted run does not replace the good parquet.

In [ ]:
from pathlib import Path
import subprocess, sys

HERE = Path.cwd().resolve()
if HERE.name != "Daily Screener":
    HERE = HERE / "Daily Screener"
assert (HERE / "update_tf15_parquet.py").exists(), f"Open this notebook from the ISTL repository: {HERE}"

command = [sys.executable, str(HERE / "update_tf15_parquet.py"), "--period", "60d", "--pause", "0.15"]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import pandas as pd
data_path = HERE.parent / "Kronos IDX FineTune 15 Minutes/data/idx_kronos_all_15m.parquet"
prices = pd.read_parquet(data_path)
print({
    "rows": len(prices), "tickers": prices.ticker.nunique(),
    "first_bar": str(prices.date.min()), "last_completed_bar": str(prices.date.max()),
    "duplicates": int(prices.duplicated(["ticker", "date"]).sum()),
})
prices.tail()

## Push only the refreshed parquet

This cell commits and pushes only the canonical TF15 parquet. Other dirty or
staged project files are excluded from the commit by an explicit pathspec.

In [ ]:
import importlib.util

push_module = HERE / "push_tf15_parquet.py"
spec = importlib.util.spec_from_file_location("tf15_push", push_module)
tf15_push = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tf15_push)
tf15_push.push_parquet()